# Imports

In [1]:
import sys, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if ROOT not in sys.path:
    sys.path.append(ROOT)

import random
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, classification_report
)
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from modules.eml_loader import EmlLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

/home/defalt/.pyenv/versions/venv-3.10.12/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cuda


# Config

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "microsoft/deberta-v3-large"
CSV_PATH = "../../datasets/processed/final_combined.csv"

# Load Dataset

In [3]:
df = pd.read_csv("../../datasets/processed/final_combined.csv")
headers_df = df[["headers_raw","label"]].copy()

# Remove empty/missing
headers_df["headers_raw"] = headers_df["headers_raw"].fillna("").astype(str)
headers_df = headers_df[headers_df["headers_raw"].str.strip().astype(bool)]

# Drop duplicates
headers_df = headers_df.drop_duplicates(subset=["headers_raw"])

print("Header-only rows:", headers_df.shape[0])
print(headers_df["label"].value_counts(normalize=True))

train_df, val_df = train_test_split(
    headers_df,
    test_size=0.1,
    random_state=SEED,
    stratify=headers_df["label"]
)

print("Train size:", len(train_df), "Val size:", len(val_df))
print("Train label counts:", train_df["label"].value_counts())
print("Val label counts:", val_df["label"].value_counts())

Header-only rows: 82915
label
1    0.634155
0    0.365845
Name: proportion, dtype: float64
Train size: 74623 Val size: 8292
Train label counts: label
1    47323
0    27300
Name: count, dtype: int64
Val label counts: label
1    5258
0    3034
Name: count, dtype: int64


/tmp/ipykernel_248880/405408840.py:1: DtypeWarning: Columns (3,4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../datasets/processed/final_combined.csv")


# Tokenizer + Dataset

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Assigned pad_token =", tokenizer.pad_token)

MAX_LEN = 512

def tokenize_headers(batch):
    return tokenizer(
        batch["headers_raw"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = Dataset.from_pandas(train_df.rename(columns={"label":"labels"}))
val_ds   = Dataset.from_pandas(val_df.rename(columns={"label":"labels"}))

train_ds_tokenized = train_ds.map(tokenize_headers, batched=True, remove_columns=["headers_raw"])
val_ds_tokenized   = val_ds.map(tokenize_headers, batched=True, remove_columns=["headers_raw"])

train_ds_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer)

/home/defalt/.pyenv/versions/venv-3.10.12/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map: 100%|██████████| 8292/8292 [00:01<00:00, 5627.85 examples/s]


# Model

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id
# model.gradient_checkpointing_enable()  # Gradient checkpointing
model.to(DEVICE)

print("Model loaded:", MODEL_NAME)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: microsoft/deberta-v3-large


# Metrics

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)

    print("\nConfusion Matrix:")
    print(confusion_matrix(labels, preds))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Trainning Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="header_deberta_large_out",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.06,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    bf16=True,
    fp16=False,
    max_grad_norm=1.0,
    gradient_checkpointing=False,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    optim="adamw_torch",
    label_smoothing_factor=0.1,
    report_to="none",
)

# Trainer

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_tokenized,
    eval_dataset=val_ds_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_248880/2193099151.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Train

In [9]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# Save Model

In [ ]:
trainer.save_model("header_deberta_large")
tokenizer.save_pretrained("header_deberta_large_tokenizer")
print("Training complete and model saved.")

# Confusion Matrix

In [ ]:
preds = trainer.predict(val_ds_tokenized)
y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit","Phishing"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Headers DeBERTa v3-Large")
plt.show()

# Loss Curve & Metrics Curve

In [ ]:
df_logs = pd.DataFrame(trainer.state.log_history)
loss_df = df_logs[df_logs["loss"].notna()]
plt.figure(figsize=(10,5))
plt.plot(loss_df["step"], loss_df["loss"], marker="o")
plt.title("Training Loss Curve")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Metrics per epoch
eval_df = df_logs[df_logs["eval_loss"].notna()].copy()
if "epoch" not in eval_df.columns:
    eval_df["epoch"] = range(1, len(eval_df)+1)

plt.figure(figsize=(14,6))
for col, label in [
    ("eval_accuracy", "Accuracy"),
    ("eval_precision", "Precision"),
    ("eval_recall", "Recall"),
    ("eval_f1", "F1 Score")
]:
    if col in eval_df:
        plt.plot(eval_df["epoch"], eval_df[col], marker="o", label=label)

plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.title("Evaluation Metrics per Epoch")
plt.legend()
plt.grid(True)
plt.show()

# Precision-Recall Curve

In [ ]:
probs = torch.softmax(torch.tensor(preds.predictions), dim=1)[:, 1].numpy()
prec, rec, thresh = precision_recall_curve(y_true, probs)

plt.figure(figsize=(10,5))
plt.plot(rec, prec)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.show()

# Classification Report

In [ ]:
final_preds = preds.predictions.argmax(-1)
print(classification_report(y_true, final_preds, target_names=["Legit","Phishing"]))

# Inference Helper

In [ ]:
def predict_header_eml(eml_path: str):
    loader = EmlLoader()
    _, _, headers_raw = loader.separate_header_blocks(eml_path)
    if not headers_raw.strip():
        return {"error": "Cannot read headers from EML."}

    model.eval()
    inputs = tokenizer(
        [headers_raw],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
        pred   = int(np.argmax(probs))

    return {
        "prediction": "PHISHING" if pred==1 else "LEGIT",
        "prob_legit": float(probs[0]),
        "prob_phishing": float(probs[1]),
    }

FILE = "4s.eml"
for key, value in predict_header_eml(FILE).items():
    print(f"{key}: {value}")